In [8]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [16]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke({"query": "Rajasthani recipes chickpea flour, tomatoes, curd in Hindi"})

{'query': 'Rajasthani recipes chickpea flour, tomatoes, curd in Hindi',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.instagram.com/reel/DI1MXLGpDDL?hl=en',
   'title': 'Rajasthani style Sabzi: Veggies in yogurt and besan. One ...',
   'content': 'Here we roast the chickpea flour and mix it with yogurt. Recipe: 1/3 cup roasted chickpea flour … uses fresh spices, tomatoes and slow cooked.',
   'score': 0.73914057,
   'raw_content': None},
  {'url': 'https://www.facebook.com/100051311678691/posts/rajasthani-kadhi-recipe-marwadi-kadhi-besan-kadhi/1344006707319706',
   'title': 'Rajasthani kadhi recipe | Marwadi kadhi | besan kadhi',
   'content': 'Ingredients - Gram Flour/Besan/Chickpea flour - 1 cup Curd - 1/2 cup for batter + 2/3 cup for gravy Salt as per taste for both dough and masala ...Read more',
   'score': 0.69968605,
   'raw_content': None},
  {'url': 'https://www.facebook.com/Nehacookbook/posts/%E0%AA%B2%E0%AA%96%E0%AB%87%E0%AA%

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [30]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import os

model = init_chat_model(
    model="arc:lite",
    model_provider="openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),
)
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [52]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "3"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover potatoes, tomatoes, peanuts and rice. What can I make?")]},
    config
)

print(response)

{'messages': [HumanMessage(content='I have some leftover potatoes, tomatoes, peanuts and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='28dd1efe-952a-4fce-9826-673c1a11acaa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 139, 'total_tokens': 160, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'arc:lite', 'system_fingerprint': 'vllm-0.23.0-406383e7', 'id': 'chatcmpl-8c6afccdde0f761a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5c6d-da93-7193-b4e2-c33db744accc-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes with potatoes tomatoes peanuts and rice'}, 'id': 'chatcmpl-tool-bb105f8a4d0b0764', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 139, 'output_tokens': 21, 'total_tokens': 160, 'input_token_details': {}, 'output_token_details': {}}), Tool

In [53]:
from pprint import pprint

pprint(response["messages"][-1].content)

('Since you have a unique combination of starches (potatoes and rice) and '
 'textures (creamy peanuts and acidic tomatoes), you can go in a few delicious '
 'directions. \n'
 '\n'
 'Based on your ingredients, here are three different "vibes" I suggest:\n'
 '\n'
 '### 1. The "Comfort Bowl": Creamy Tomato & Peanut Potato Curry\n'
 'This is a hearty, warming dish. The potatoes and tomatoes create a thick, '
 'savory sauce, and the peanuts add a delightful crunch and richness.\n'
 '*   **How it works:** You would simmer the diced potatoes and tomatoes '
 'together with some spices (like curry powder or cumin if you have them) '
 'until soft, then stir in crushed peanuts (or a spoonful of peanut butter if '
 'you have it) to create a creamy sauce. Serve this generously over your '
 'cooked rice.\n'
 '\n'
 '### 2. The "Flavor Bomb": Curried Tomato Peanut Rice\n'
 'This is a quicker, one-pot style meal where the rice becomes the star.\n'
 '*   **How it works:** Sauté your diced potatoes unti